# 5 본 실행 — LightGBM 순위 회귀 (로컬 CPU)

- **main_v2(2026-11-23)**: 설정은 main_v1과 같고 입력만 추가 — 업종 피처 6개(자산성장률·GP/A·CF/P·S/P·비율 발생액·매출성장률), H2 확장 5개(debt_change, policy_change, rate_change, 두 상호작용). 부스팅 기본 세트 24개, 주 세트 38개.
- **설정(config `main_run`, 2026-11-22 튜닝 결정)**: 학습률 0.01, 깊이 3, 트리 500 고정, XGB L2 0, pairwise 표본 8개 쌍(seed 0). 입력 세트 6개, 16 Fold, 평균 채움, 순위 변환.
- **모델 저장**: Fold × 세트마다 예측에 쓴 모델(고른 H3 λ, 트리 500개) → 결과 폴더 `models.pkl`.
- **SHAP**: 기본 세트·주 세트, 각 Fold 모델로 그 Fold 바깥 검증 행. 주 세트는 피처 쌍 상호작용 요약도. 계산마다 저장 모델 예측 = 실행 때 예측, SHAP 합 = 예측을 확인하고 어긋나면 멈춘다.
- **중간 저장**: 단위(모델 × 세트, SHAP은 모델 × SHAP 세트)마다 `model_results/<AUTHOR>/main_v2_parts/`. 끊기면 Run All로 이어감.
- **결과**: `model_results/<AUTHOR>/<시각>_main_v2_<모델>_basefeat/` — 보고 표·`shap_importance.csv`·`shap_interactions.csv`(result.zip), `models.pkl`·`shap_values.parquet`·예측값(각자 PC).
- 예상: 학습 약 30분 + SHAP(smoke로 잰 값은 아래 셀 주석).

## 1. 설정

In [ ]:
import os, sys, subprocess
from datetime import datetime
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/JiunShin1214/finance_analytics.git'
GIT_REF = ''                    # Colab: 브랜치 이름(예: 'jiun') — 그 브랜치의 GitHub 최신 코드
AUTHOR = 'jiun'                 # 결과 폴더 model_results/<AUTHOR>/ (영문·숫자·_·-만)
RUN_NAME = 'main_v2'        # 같은 이름으로 다시 Run All 하면 Drive에 저장된 단위를 이어 씀
DRIVE_BASE = Path('/content/drive/MyDrive/금융데이터인텔리전스')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not GIT_REF:
        raise RuntimeError("GIT_REF에 브랜치 이름을 적는다.")
    ROOT = Path('/content/finance_analytics')
    if not ROOT.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(ROOT)], check=True)
    subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', '--prune'], check=True)
    _remote = subprocess.run(['git', '-C', str(ROOT), 'rev-parse', '--verify', '--quiet', f'origin/{GIT_REF}'],
                             capture_output=True, text=True).stdout.strip()
    subprocess.run(['git', '-C', str(ROOT), 'checkout', '--detach', _remote or GIT_REF], check=True)
    print('받은 코드:', GIT_REF, '@', subprocess.run(['git', '-C', str(ROOT), 'rev-parse', '--short', 'HEAD'],
                                               capture_output=True, text=True, check=True).stdout.strip())
    if not (ROOT / 'data').is_symlink():       # 저장소 data/를 Drive 공유 data로 연결
        (ROOT / 'data').rename(ROOT / 'data_repo_placeholder')
        (ROOT / 'data').symlink_to(DRIVE_BASE / 'data')
    OUT_ROOT = Path('/content/drive/MyDrive/model_results')   # 중간 저장(parts)과 결과 모두 내 드라이브 → 세션이 끊겨도 남음
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir() and (p / 'config' / 'experiment.yaml').is_file())
    OUT_ROOT = None

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import yaml
import pandas as pd
from src.evaluation import model_test as mt
from src.evaluation import base_model_runs as bmr
from src.features.h3 import compute_h3_sigma_grid

config = yaml.safe_load((ROOT / 'config' / 'experiment.yaml').read_text(encoding='utf-8'))
mt.check_no_test(config)
need = [config['PATHS'][k] for k in ('panel', 'eligibility', 'h1_components', 'base_features', 'h2', 'h4')]
missing = [n for n in need if not (ROOT / n).is_file()]
if missing:
    raise FileNotFoundError('Drive data/에 없음: ' + ', '.join(missing))
STARTED = datetime.now()
RUN = bmr.main_run_spec(config)      # config BASE_FEATURES.model_input.main_run (튜닝 결정 2026-11-22)
mt.boosting.MAX_TREES = RUN['trees']  # 트리 500 고정: 상한도 500(같은 학습의 첫 500개 트리라 결과는 같고 시간만 준다)
print('설정:', {k: RUN[k] for k in ('depth', 'learning_rate', 'trees', 'xgb_l2')}, '| pairwise 쌍:', RUN['pairwise_pairs'])
PARTS = bmr.parts_dir(ROOT, config, author=AUTHOR, run_name=RUN_NAME, out_root=OUT_ROOT)
print('중간 저장 폴더:', PARTS)
print('이미 저장된 단위:', len(list(PARTS.glob('*.pkl'))))

## 2. H3 격자·전처리

In [ ]:
panel = pd.read_parquet(ROOT / config['PATHS']['panel'], columns=['stock_id', 'month', 'return', 'eligible'])
eligibility = pd.read_parquet(ROOT / config['PATHS']['eligibility'])
H3_GRID = compute_h3_sigma_grid(panel, eligibility)
del panel, eligibility
PREP = bmr.prepare(ROOT, config, 'boosting', input_transform='rank', h3_grid=H3_GRID)
_sets = bmr.input_sets('boosting')
assert {'rmw', 'gpa', 'cfp', 'asset_growth'} <= set(_sets['base']), '업종 피처(2026-11-23)가 없는 코드다'
assert {'debt_change', 'rate_change', 'policy_change_x_debt'} <= set(_sets['base+h1+h3+h2+h4']), 'H2 확장(2026-11-23)이 없는 코드다'
print({k: len(v) for k, v in _sets.items()})   # 기대: 기본 24, 주 세트 38

## 3. 학습 (6단위: 세트 6개)

In [ ]:
KINDS = ['lgb_rankreg']
fp = bmr.fingerprint(ROOT, config, 'boosting', 'rank')
RESULTS = bmr.run_units(bmr.main_units(PREP, config, KINDS), PARTS, fp)

## 4. SHAP (2단위: 기본 세트, 주 세트 + 상호작용)

In [ ]:
SHAP = bmr.run_units(bmr.shap_units(PREP, config, RESULTS, device='cpu'), PARTS, fp)

## 5. 요약

In [ ]:
rows = []
for unit, r in RESULTS.items():
    kind, set_name = unit.split('__')
    vk, sel = r['val_keys'], r['selections']
    (key, pred), = r['predictions'].items()
    ic = mt.monthly_ic(vk.target_month.to_numpy(), pred, vk.y.to_numpy()).ic
    rows.append({'모델': kind, '세트': set_name, '평균 IC': ic.mean(), 'ICIR': ic.mean() / ic.std(ddof=1),
                 '학습 IC': sel.train_ic.mean(), '학습−검증': sel.train_ic.mean() - ic.mean(), '개월': int(ic.notna().sum()),
                 '모델 수': len(r.get('models', [])), '학습 시간(분)': r['timings'].seconds.sum() / 60})
T = pd.DataFrame(rows)
print(T.round(4).to_string())
for unit, s in SHAP.items():
    imp = s['importance']
    top = imp[imp.fold.eq('all')].sort_values('mean_abs', ascending=False).head(8)
    print(f'\n{unit} — 평균 |SHAP| 상위 8:'); print(top[['feature', 'mean_abs', 'share', 'corr_input']].round(4).to_string(index=False))
    it = s['interactions']
    if len(it):
        it = it[it.fold.eq('all') & it.feature_a.ne(it.feature_b)].sort_values('mean_abs', ascending=False).head(5)
        print('상호작용 상위 5:'); print(it[['feature_a', 'feature_b', 'mean_abs', 'mean']].round(5).to_string(index=False))

## 6. 저장

In [ ]:
SAVED = bmr.finalize_main(ROOT, config, PREP, RESULTS, SHAP, author=AUTHOR, run_name=RUN_NAME, started=STARTED, parts=PARTS,
                          out_root=OUT_ROOT)
SAVED